In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_logs_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (metric_df, trace_df, log_df, error_logs_df):
    df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Prepare summaries for each file

# metric.csv: rows, min/max ts, top up to 50 unique (cmdb_id, kpi_name) pairs with counts
metric_rows = metric_df.shape[0]
metric_min_ts = metric_df['dt'].min()
metric_max_ts = metric_df['dt'].max()
metric_top_pairs = (
    metric_df.groupby(['cmdb_id', 'kpi_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# trace.csv: rows, min/max ts, top up to 50 unique (cmdb_id, trace_name) pairs with counts
trace_rows = trace_df.shape[0]
trace_min_ts = trace_df['dt'].min()
trace_max_ts = trace_df['dt'].max()
trace_top_pairs = (
    trace_df.groupby(['cmdb_id', 'trace_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# log.csv: rows, min/max ts, top up to 50 unique (cmdb_id, log_name) pairs with counts
log_rows = log_df.shape[0]
log_min_ts = log_df['dt'].min()
log_max_ts = log_df['dt'].max()
log_top_pairs = (
    log_df.groupby(['cmdb_id', 'log_name'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# error_logs.csv: rows, min/max ts, top up to 50 unique cmdb_id values with counts
error_rows = error_logs_df.shape[0]
error_min_ts = error_logs_df['dt'].min()
error_max_ts = error_logs_df['dt'].max()
error_top_cmdb = (
    error_logs_df.groupby(['cmdb_id'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(50)
    .reset_index(drop=True)
)

# overall set of unique cmdb_id across all files
all_cmdb_ids = pd.Series(
    sorted(
        set(metric_df['cmdb_id'].dropna().unique())
        | set(trace_df['cmdb_id'].dropna().unique())
        | set(log_df['cmdb_id'].dropna().unique())
        | set(error_logs_df['cmdb_id'].dropna().unique())
    ),
    name='cmdb_id'
)

# Compact summary table of file-level counts and timestamp ranges
file_summary = pd.DataFrame([
    {"file":"metric.csv",     "rows": metric_rows, "min_ts_utc": metric_min_ts, "max_ts_utc": metric_max_ts},
    {"file":"trace.csv",      "rows": trace_rows, "min_ts_utc": trace_min_ts, "max_ts_utc": trace_max_ts},
    {"file":"log.csv",        "rows": log_rows, "min_ts_utc": log_min_ts, "max_ts_utc": log_max_ts},
    {"file":"error_logs.csv", "rows": error_rows, "min_ts_utc": error_min_ts, "max_ts_utc": error_max_ts},
])

# Display compact results (variables; IPython will show them)
file_summary, metric_top_pairs, trace_top_pairs, log_top_pairs, error_top_cmdb, all_cmdb_ids

```
Out[1]:
```
# Summarize the previously computed results in plain English (compact)
import pandas as pd

def fmt_ts(x):
    return x.isoformat() if pd.notna(x) else "None"

metric_rows = file_summary.loc[file_summary.file=="metric.csv","rows"].iat[0]
metric_min = file_summary.loc[file_summary.file=="metric.csv","min_ts_utc"].iat[0]
metric_max = file_summary.loc[file_summary.file=="metric.csv","max_ts_utc"].iat[0]
metric_unique_pairs = metric_df.groupby(['cmdb_id','kpi_name']).size().shape[0]
metric_top5 = [
    f"{r.cmdb_id}|{r.kpi_name}:{r['count']}"
    for r in metric_top_pairs.head(5).to_dict('records')
]

trace_rows = file_summary.loc[file_summary.file=="trace.csv","rows"].iat[0]

log_rows = file_summary.loc[file_summary.file=="log.csv","rows"].iat[0]
log_min = file_summary.loc[file_summary.file=="log.csv","min_ts_utc"].iat[0]
log_max = file_summary.loc[file_summary.file=="log.csv","max_ts_utc"].iat[0]
log_unique_pairs = log_df.groupby(['cmdb_id','log_name']).size().shape[0]
log_top5 = [
    f"{r.cmdb_id}|{r.log_name}:{r['count']}"
    for r in log_top_pairs.head(5).to_dict('records')
]

error_rows = file_summary.loc[file_summary.file=="error_logs.csv","rows"].iat[0]

all_ids = all_cmdb_ids.tolist()
all_ids_count = len(all_ids)

summary = (
    f"Summary of telemetry files:\n"
    f"- metric.csv: {metric_rows} rows; time range {fmt_ts(metric_min)} to {fmt_ts(metric_max)} UTC; "
    f"{metric_unique_pairs} unique (cmdb_id, kpi_name) pairs. Top up to 5 pairs: {metric_top5}.\n"
    f"- trace.csv: {trace_rows} rows (no trace data available).\n"
    f"- log.csv: {log_rows} rows; time range {fmt_ts(log_min)} to {fmt_ts(log_max)} UTC; "
    f"{log_unique_pairs} unique (cmdb_id, log_name) pairs. Top up to 5 pairs: {log_top5}.\n"
    f"- error_logs.csv: {error_rows} rows (no error log entries).\n"
    f"- Overall unique cmdb_id across all files: {all_ids_count} ids: {all_ids}.\n"
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(             file  rows                min_ts_utc                max_ts_utc
0      metric.csv  2000 2024-01-19 06:04:00+00:00 2024-01-19 06:28:00+00:00
1       trace.csv     0                       NaT                       NaT
2         log.csv   458 2024-01-19 06:04:00+00:00 2024-01-19 06:28:00+00:00
3  error_logs.csv     0                       NaT                       NaT,          cmdb_id    kpi_name  count
0          carts         cpu     25
1          carts       error     25
2          carts  latency-50     25
3          carts  latency-90     25
4          carts         mem     25
..           ...         ...    ...
45       payment         mem     25
46       payment      socket     25
47       payment    workload     25
48  queue-master         cpu     25
49  queue-master      diskio     25

[50 rows x 3 columns], Empty DataFrame
Columns: [cmdb_id, trace_name, count]
Index: [],        cmdb_id         log_name  count
0     carts-db    log.row_count     25
1     carts-db  log.error_count     25
2    catalogue  log.error_count     25
3    catalogue    log.row_count     25
4    front-end    log.row_count     25
..         ...              ...    ...
17       carts    log.row_count     23
18  session-db  log.error_count      5
19  session-db    log.row_count      5
20     user-db  log.error_count      1
21     user-db    log.row_count      1

[22 rows x 3 columns], Empty DataFrame
Columns: [cmdb_id, count]
Index: [], 0                 carts
1              carts-db
2             catalogue
3          catalogue-db
4             front-end
5                orders
6             orders-db
7               payment
8          queue-master
9              rabbitmq
10    rabbitmq-exporter
11           session-db
12             shipping
13                 user
14              user-db
Name: cmdb_id, dtype: object)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

# Reuse existing metric_df (already has 'dt' parsed to UTC) and other stateful variables.
import pandas as pd

# 1) Compute global P95 per (cmdb_id, kpi_name) using the entire metric series
p95_series = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='series_global_p95')
)

# 2) Filter metric.csv to the incident window (UTC)
start_ts = pd.Timestamp("2024-01-19 06:01:32", tz="UTC")
end_ts   = pd.Timestamp("2024-01-19 06:31:32", tz="UTC")
metric_window = metric_df[(metric_df['dt'] >= start_ts) & (metric_df['dt'] <= end_ts)].copy()

# 3) For filtered points, join with series P95 and mark exceedances (value >= series_global_p95)
metric_window = metric_window.merge(p95_series, on=['cmdb_id','kpi_name'], how='left')
metric_window['exceed'] = metric_window['value'] >= metric_window['series_global_p95']

# 4) Aggregate per series in window
agg = (
    metric_window
    .groupby(['cmdb_id','kpi_name'], as_index=False)
    .agg(
        total_points_in_window = ('value','size'),
        count_exceedances = ('exceed','sum'),
        earliest_exceedance_timestamp_utc = ('dt', lambda s: s[metric_window.loc[s.index,'exceed']].min() if any(metric_window.loc[s.index,'exceed']) else pd.NaT),
        max_value_in_window = ('value','max'),
        series_global_p95 = ('series_global_p95','first')
    )
)

# Ensure earliest_exceedance_timestamp_utc is timezone-aware datetime or NaT; convert via apply
agg['earliest_exceedance_timestamp_utc'] = pd.to_datetime(agg['earliest_exceedance_timestamp_utc'], utc=True)

# Filter to only series that had at least one exceedance for ranking, but keep full list for counts
agg_with_exceed = agg[agg['count_exceedances']>0].copy()

# Sort by count_exceedances desc, then by total_points desc, then by max_value desc
agg_with_exceed = agg_with_exceed.sort_values(
    ['count_exceedances','total_points_in_window','max_value_in_window'],
    ascending=[False, False, False]
).reset_index(drop=True)

# Top 20 rows as requested
top20 = agg_with_exceed.head(20).loc[:,[
    'cmdb_id','kpi_name','total_points_in_window','count_exceedances',
    'earliest_exceedance_timestamp_utc','max_value_in_window','series_global_p95'
]]

# 5) Totals: total number of distinct series examined (in metric.csv) and how many had at least one exceedance in the window
total_distinct_series = metric_df.groupby(['cmdb_id','kpi_name']).ngroups
series_with_at_least_one_exceedance = agg_with_exceed.shape[0]
# Also show how many distinct series appeared in the window (context)
distinct_series_in_window = metric_window.groupby(['cmdb_id','kpi_name']).ngroups

# Compact summary as small DataFrame for display
totals = pd.DataFrame([
    {'metric': 'total_distinct_series_in_metric_csv', 'value': total_distinct_series},
    {'metric': 'distinct_series_present_in_window', 'value': distinct_series_in_window},
    {'metric': 'series_with_at_least_one_exceedance_in_window', 'value': series_with_at_least_one_exceedance}
])

# Display results (IPython will show these variables)
top20, totals

```
Out[2]:
```
# Create a plain-English summary string from previous results (compact)
rows = []
for i, r in top20.reset_index(drop=True).iterrows():
    ts = r['earliest_exceedance_timestamp_utc']
    ts_str = ts.isoformat() if pd.notna(ts) else "None"
    rows.append(
        f"{i+1}. {r['cmdb_id']} | {r['kpi_name']} — points:{int(r['total_points_in_window'])}, "
        f"exceedances:{int(r['count_exceedances'])}, earliest:{ts_str}, "
        f"max:{r['max_value_in_window']}, P95:{r['series_global_p95']}"
    )

summary = (
    "Metric exceedance summary for window 2024-01-19 06:01:32 to 2024-01-19 06:31:32 UTC:\n"
    f"- Total distinct (cmdb_id, kpi_name) series in metric.csv: {int(totals.loc[0,'value'])}\n"
    f"- Distinct series present in the window: {int(totals.loc[1,'value'])}\n"
    f"- Series with >=1 exceedance in the window: {int(totals.loc[2,'value'])}\n\n"
    "Top series by number of exceedances (up to 20):\n" + "\n".join(rows)
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id kpi_name  total_points_in_window  count_exceedances earliest_exceedance_timestamp_utc  max_value_in_window  series_global_p95
0   rabbitmq-exporter      mem                      25                 25         2024-01-19 06:04:00+00:00            4575232.0          4575232.0
1            rabbitmq   socket                      25                 25         2024-01-19 06:04:00+00:00                 11.0               11.0
2           orders-db   socket                      25                 25         2024-01-19 06:04:00+00:00                  8.0                8.0
3             payment   socket                      25                 25         2024-01-19 06:04:00+00:00                  6.0                6.0
4        catalogue-db   socket                      25                 25         2024-01-19 06:04:00+00:00                  4.0                4.0
5        queue-master   socket                      25                 25         2024-01-19 06:04:00+00:00                  3.0                3.0
6          session-db   socket                      25                 25         2024-01-19 06:04:00+00:00                  3.0                3.0
7   rabbitmq-exporter   socket                      25                 25         2024-01-19 06:04:00+00:00                  1.0                1.0
8               carts    error                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
9           catalogue   diskio                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
10          catalogue    error                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
11       catalogue-db   diskio                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
12          front-end    error                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
13             orders    error                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
14            payment    error                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
15           rabbitmq   diskio                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
16               user    error                      25                 25         2024-01-19 06:04:00+00:00                  0.0                0.0
17               user   socket                      25                 20         2024-01-19 06:04:00+00:00                 24.0               24.0
18            user-db   socket                      25                 20         2024-01-19 06:04:00+00:00                 13.0               13.0
19           carts-db   socket                      25                  6         2024-01-19 06:23:00+00:00                 29.0               29.0,                                           metric  value
0            total_distinct_series_in_metric_csv     80
1              distinct_series_present_in_window     80
2  series_with_at_least_one_exceedance_in_window     80)```
```

In [ ]:
# In[3]:

# Reuse existing loaded DataFrames: metric_df, log_df (they already have 'dt' parsed to UTC)
import pandas as pd

# Candidate services
candidates = ["carts","catalogue","orders","payment","user"]

# Ensure incident window (UTC)
start_ts = pd.Timestamp("2024-01-19 06:01:32", tz="UTC")
end_ts   = pd.Timestamp("2024-01-19 06:31:32", tz="UTC")

# 1) Compute global P95 for (cmdb_id,kpi_name) among candidate cmdb_ids using full metric_df
metric_candidates_full = metric_df[metric_df['cmdb_id'].isin(candidates)]
p95_candidates = (
    metric_candidates_full
    .groupby(['cmdb_id','kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='series_global_p95')
)

# 2) Filter metric.csv to the incident window and limit to candidate cmdb_ids
metric_window_candidates = metric_df[
    (metric_df['dt'] >= start_ts) & (metric_df['dt'] <= end_ts) & (metric_df['cmdb_id'].isin(candidates))
].copy()

# Merge P95 thresholds (computed globally) into windowed points
metric_window_candidates = metric_window_candidates.merge(
    p95_candidates, on=['cmdb_id','kpi_name'], how='left'
)

# Mark exceedances (value >= series_global_p95). If p95 is NaN (no series in full data), treat as False
metric_window_candidates['exceed'] = metric_window_candidates['value'] >= metric_window_candidates['series_global_p95']

# Aggregate per (cmdb_id, kpi_name) for candidates
def agg_metric_group(g):
    total = int(g.shape[0])
    count_exc = int(g['exceed'].sum())
    earliest_exc = g.loc[g['exceed'], 'dt'].min() if count_exc>0 else pd.NaT
    maxv = float(g['value'].max()) if total>0 else float('nan')
    p95 = float(g['series_global_p95'].iloc[0]) if 'series_global_p95' in g.columns else float('nan')
    return pd.Series({
        'total_points_in_window': total,
        'count_exceedances': count_exc,
        'earliest_exceedance_timestamp_utc': earliest_exc,
        'max_value_in_window': maxv,
        'series_global_p95': p95
    })

if metric_window_candidates.shape[0] > 0:
    metric_agg = metric_window_candidates.groupby(['cmdb_id','kpi_name']).apply(agg_metric_group).reset_index()
else:
    # No rows in window for candidates; create empty DataFrame with expected columns
    metric_agg = pd.DataFrame(columns=['cmdb_id','kpi_name','total_points_in_window','count_exceedances',
                                       'earliest_exceedance_timestamp_utc','max_value_in_window','series_global_p95'])

# For completeness, also include series that exist in full data but had zero points in window:
# find candidate series from p95_candidates that are not in metric_agg, and add rows with zeros/NaT
existing_pairs = set(metric_agg.apply(lambda r: (r['cmdb_id'], r['kpi_name']), axis=1).tolist())
for _, row in p95_candidates.iterrows():
    pair = (row['cmdb_id'], row['kpi_name'])
    if pair not in existing_pairs:
        metric_agg = pd.concat([metric_agg, pd.DataFrame([{
            'cmdb_id': row['cmdb_id'],
            'kpi_name': row['kpi_name'],
            'total_points_in_window': 0,
            'count_exceedances': 0,
            'earliest_exceedance_timestamp_utc': pd.NaT,
            'max_value_in_window': float('nan'),
            'series_global_p95': float(row['series_global_p95'])
        }])], ignore_index=True)

# Sort metric_agg for readability: by count_exceedances desc, then total points desc
metric_agg = metric_agg.sort_values(['count_exceedances','total_points_in_window'], ascending=[False,False]).reset_index(drop=True)

# 3) Log.csv analysis for the same window and candidate cmdb_ids
log_window = log_df[(log_df['dt'] >= start_ts) & (log_df['dt'] <= end_ts) & (log_df['cmdb_id'].isin(candidates))].copy()

# Prepare candidate rows DataFrame
cand_df = pd.DataFrame({'cmdb_id': candidates})

# number_of_rows_present_in_window
rows_count = log_window.groupby('cmdb_id').size().reset_index(name='number_of_rows_present_in_window')

# log.error_count aggregates
error_df = log_window[log_window['log_name']=='log.error_count'].copy()
if error_df.shape[0] > 0:
    error_summax = error_df.groupby('cmdb_id').agg(
        sum_log_error_count = ('value','sum'),
        max_log_error_count = ('value','max')
    ).reset_index()
    earliest_error_pos = error_df[error_df['value']>0].groupby('cmdb_id')['dt'].min().reset_index(name='earliest_timestamp_where_log.error_count_gt0')
else:
    error_summax = pd.DataFrame(columns=['cmdb_id','sum_log_error_count','max_log_error_count'])
    earliest_error_pos = pd.DataFrame(columns=['cmdb_id','earliest_timestamp_where_log.error_count_gt0'])

# log.row_count aggregates
rowcount_df = log_window[log_window['log_name']=='log.row_count'].copy()
if rowcount_df.shape[0] > 0:
    rowcount_summax = rowcount_df.groupby('cmdb_id').agg(
        sum_log_row_count = ('value','sum'),
        max_log_row_count = ('value','max')
    ).reset_index()
else:
    rowcount_summax = pd.DataFrame(columns=['cmdb_id','sum_log_row_count','max_log_row_count'])

# Merge everything into a compact per-candidate table
log_agg = cand_df.merge(rows_count, on='cmdb_id', how='left') \
                 .merge(error_summax, on='cmdb_id', how='left') \
                 .merge(earliest_error_pos, on='cmdb_id', how='left') \
                 .merge(rowcount_summax, on='cmdb_id', how='left')

# Fill NaNs: counts/sums/max -> 0, earliest timestamp keep NaT
num_cols = ['number_of_rows_present_in_window','sum_log_error_count','max_log_error_count','sum_log_row_count','max_log_row_count']
for c in num_cols:
    if c in log_agg.columns:
        log_agg[c] = log_agg[c].fillna(0).astype(int)

# Ensure earliest timestamp column is timezone-aware or NaT
if 'earliest_timestamp_where_log.error_count_gt0' in log_agg.columns:
    log_agg['earliest_timestamp_where_log.error_count_gt0'] = pd.to_datetime(log_agg['earliest_timestamp_where_log.error_count_gt0'], utc=True)

# Reorder columns for compactness
log_agg = log_agg[[
    'cmdb_id',
    'number_of_rows_present_in_window',
    'sum_log_error_count',
    'max_log_error_count',
    'earliest_timestamp_where_log.error_count_gt0',
    'sum_log_row_count',
    'max_log_row_count'
]]

# Final outputs:
# - metric_agg: one row per (cmdb_id,kpi_name) for candidate services with requested metrics
# - log_agg: one row per candidate cmdb_id with requested log statistics

# Keep outputs compact: show up to 100 rows total (these are small); display metric_agg and log_agg
metric_agg, log_agg

```
Out[3]:
```
# Summarize results for the candidate services in plain English using existing variables
import pandas as pd

candidates = ["carts","catalogue","orders","payment","user"]

# Use metric_agg and log_agg from previous execution
# Aggregate per candidate for metrics
metric_summary_rows = []
for svc in candidates:
    df = metric_agg[metric_agg['cmdb_id']==svc]
    total_series = int(df.shape[0])
    series_with_exceed = int((df['count_exceedances']>0).sum())
    earliest_exc = df['earliest_exceedance_timestamp_utc'].min() if series_with_exceed>0 else pd.NaT
    max_val = df['max_value_in_window'].max() if total_series>0 else float('nan')
    metric_summary_rows.append({
        'cmdb_id': svc,
        'total_metric_series': total_series,
        'series_with_exceedances': series_with_exceed,
        'earliest_exceedance_utc': earliest_exc,
        'max_value_in_window_over_series': max_val
    })

metric_summary_df = pd.DataFrame(metric_summary_rows)

# Prepare log summary (log_agg already per candidate)
log_summary_df = log_agg.set_index('cmdb_id').reindex(candidates).reset_index()

# Build a concise plain-English summary string
lines = []
lines.append("Summary for candidate services (window 2024-01-19 06:01:32 to 2024-01-19 06:31:32 UTC):\n")

for _, r in metric_summary_df.iterrows():
    svc = r['cmdb_id']
    e_ts = r['earliest_exceedance_utc']
    e_str = e_ts.isoformat() if pd.notna(e_ts) else "None"
    lines.append(
        f"- {svc} (metrics): examined {r['total_metric_series']} series; "
        f"{r['series_with_exceedances']} series had >=1 exceedance; "
        f"earliest exceedance: {e_str}; max metric value in window across series: {r['max_value_in_window_over_series']:.6g}"
    )

lines.append("")  # blank line

for _, r in log_summary_df.iterrows():
    svc = r['cmdb_id']
    rows_present = int(r['number_of_rows_present_in_window'])
    sum_err = int(r['sum_log_error_count'])
    max_err = int(r['max_log_error_count'])
    earliest_err_ts = r['earliest_timestamp_where_log.error_count_gt0']
    earliest_err_str = earliest_err_ts.isoformat() if pd.notna(earliest_err_ts) else "None"
    sum_row = int(r['sum_log_row_count'])
    max_row = int(r['max_log_row_count'])
    lines.append(
        f"- {svc} (logs): rows_in_window={rows_present}; sum(log.error_count)={sum_err}; "
        f"max(log.error_count)={max_err}; earliest error>0: {earliest_err_str}; "
        f"sum(log.row_count)={sum_row}; max(log.row_count)={max_row}"
    )

lines.append("") 
lines.append("Key takeaways:")
lines.append("- Many metric series for these services show exceedances during the incident window (see counts above).")
lines.append("- Log error counts for all candidate services are zero in the window (no log.error_count > 0 recorded).")

summary_text = "\n".join(lines)
summary_text

The original code execution output of IPython Kernel is also provided below for reference:

(      cmdb_id    kpi_name  total_points_in_window  count_exceedances earliest_exceedance_timestamp_utc  max_value_in_window  series_global_p95
0       carts       error                      25                 25         2024-01-19 06:04:00+00:00         0.000000e+00       0.000000e+00
1   catalogue      diskio                      25                 25         2024-01-19 06:04:00+00:00         0.000000e+00       0.000000e+00
2   catalogue       error                      25                 25         2024-01-19 06:04:00+00:00         0.000000e+00       0.000000e+00
3      orders       error                      25                 25         2024-01-19 06:04:00+00:00         0.000000e+00       0.000000e+00
4     payment       error                      25                 25         2024-01-19 06:04:00+00:00         0.000000e+00       0.000000e+00
..        ...         ...                     ...                ...                               ...                  ...                ...
31       user         cpu                      25                  2         2024-01-19 06:15:00+00:00         1.000695e+00       9.978348e-01
32       user  latency-50                      25                  2         2024-01-19 06:18:00+00:00         5.540878e-03       5.373013e-03
33       user  latency-90                      25                  2         2024-01-19 06:18:00+00:00         1.621087e-01       1.614560e-01
34       user         mem                      25                  2         2024-01-19 06:21:00+00:00         9.002121e+06       8.993751e+06
35       user    workload                      25                  2         2024-01-19 06:16:00+00:00         1.828858e+01       1.771931e+01

[36 rows x 7 columns],      cmdb_id  number_of_rows_present_in_window  sum_log_error_count  max_log_error_count earliest_timestamp_where_log.error_count_gt0  sum_log_row_count  max_log_row_count
0      carts                                46                    0                    0                                          NaT               1352                150
1  catalogue                                50                    0                    0                                          NaT               3613                157
2     orders                                50                    0                    0                                          NaT               2660                120
3    payment                                50                    0                    0                                          NaT               2290                100
4       user                                50                    0                    0                                          NaT              12917                570)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

The Executor failed to complete the instruction, please re-write a new instruction for Executor.

```
Out[4]:
```
The Executor failed to complete the instruction, please re-write a new instruction for Executor.```
```

In [ ]:
# In[5]:

# Reuse existing metric_df (assumed loaded and with 'dt' as UTC datetime)
import pandas as pd
candidates = ["carts","catalogue","orders","payment","user"]

# Incident window (UTC)
start_ts = pd.Timestamp("2024-01-19 06:01:32", tz="UTC")
end_ts   = pd.Timestamp("2024-01-19 06:31:32", tz="UTC")

# 1) Compute global P95 for (cmdb_id,kpi_name) for candidate cmdb_ids using full metric_df
p95_candidates = (
    metric_df[metric_df['cmdb_id'].isin(candidates)]
    .groupby(['cmdb_id','kpi_name'])['value']
    .quantile(0.95)
    .reset_index(name='series_global_p95')
)

# Count of candidate series examined (unique (cmdb_id,kpi_name) among candidates in full data)
total_candidate_series_examined = int(p95_candidates.shape[0])

# 2) Filter metric_df to the incident window and candidate cmdb_ids
metric_window_candidates = metric_df[
    (metric_df['dt'] >= start_ts) & (metric_df['dt'] <= end_ts) & (metric_df['cmdb_id'].isin(candidates))
].copy()

# 3) Merge P95 thresholds into windowed points
metric_window_candidates = metric_window_candidates.merge(
    p95_candidates, on=['cmdb_id','kpi_name'], how='right'  # right join to keep series with p95 even if no window points
)

# After merge, rows coming from p95_candidates but without window points have NaNs in value/dt; keep for aggregation
# 3) Aggregate per (cmdb_id,kpi_name) compute required stats with strict '>' exceedance condition
def agg_row(g):
    # g may contain NaNs for value/dt if no points in window
    vals = g['value'].dropna()
    total_points = int(vals.shape[0])
    # strict greater than series_global_p95
    p95 = float(g['series_global_p95'].iloc[0])
    # count exceedances: for rows where value exists and > p95
    count_exc = int((vals > p95).sum())
    earliest_exc = g.loc[(g['value'] > p95) & (~g['dt'].isna()), 'dt'].min() if count_exc>0 else pd.NaT
    maxv = float(vals.max()) if total_points>0 else 0.0
    return pd.Series({
        'total_points_in_window': total_points,
        'count_exceedances': count_exc,
        'earliest_exceedance_timestamp_utc': earliest_exc,
        'max_value_in_window': maxv,
        'series_global_p95': p95
    })

metric_agg = metric_window_candidates.groupby(['cmdb_id','kpi_name'], as_index=False).apply(agg_row).reset_index(drop=True)

# 4) Exclude trivial zeros: series_global_p95==0 and max_value_in_window==0
mask_trivial = (metric_agg['series_global_p95'] == 0.0) & (metric_agg['max_value_in_window'] == 0.0)
total_candidate_series_excluded_due_to_trivial_zero = int(mask_trivial.sum())

metric_nontrivial = metric_agg[~mask_trivial].copy()

# 3b/4) Compute severity = (max_value_in_window - series_global_p95) / max(1e-9, series_global_p95)
metric_nontrivial['severity'] = (metric_nontrivial['max_value_in_window'] - metric_nontrivial['series_global_p95']) / metric_nontrivial['series_global_p95'].clip(lower=1e-9)

# 5) For each candidate service, select the single KPI with highest severity
selected_rows = []
for svc in candidates:
    svc_df = metric_nontrivial[metric_nontrivial['cmdb_id'] == svc].copy()
    if svc_df.shape[0] == 0:
        # none remain -> mark as no_nontrivial_anomalies
        selected_rows.append({
            'cmdb_id': svc,
            'kpi_name': 'no_nontrivial_anomalies',
            'total_points_in_window': 0,
            'count_exceedances': 0,
            'earliest_exceedance_timestamp_utc': pd.NaT,
            'max_value_in_window': 0.0,
            'series_global_p95': 0.0,
            'severity': 0.0
        })
    else:
        # pick row with highest severity, tie-breaker by count_exceedances desc then max_value desc
        svc_df = svc_df.sort_values(['severity','count_exceedances','max_value_in_window'], ascending=[False,False,False])
        top = svc_df.iloc[0]
        selected_rows.append({
            'cmdb_id': top['cmdb_id'],
            'kpi_name': top['kpi_name'],
            'total_points_in_window': int(top['total_points_in_window']),
            'count_exceedances': int(top['count_exceedances']),
            'earliest_exceedance_timestamp_utc': pd.to_datetime(top['earliest_exceedance_timestamp_utc'], utc=True) if pd.notna(top['earliest_exceedance_timestamp_utc']) else pd.NaT,
            'max_value_in_window': float(top['max_value_in_window']),
            'series_global_p95': float(top['series_global_p95']),
            'severity': float(top['severity'])
        })

result_df = pd.DataFrame(selected_rows, columns=[
    'cmdb_id','kpi_name','total_points_in_window','count_exceedances',
    'earliest_exceedance_timestamp_utc','max_value_in_window','series_global_p95','severity'
])

# Keep output compact: up to 5 rows (one per candidate)
summary_counts = pd.DataFrame([
    {'metric': 'total_candidate_series_examined', 'value': total_candidate_series_examined},
    {'metric': 'total_candidate_series_excluded_due_to_trivial_zero', 'value': total_candidate_series_excluded_due_to_trivial_zero}
])

# Display results (IPython will show variables)
result_df, summary_counts

```
Out[5]:
```
Summary (candidate services: carts, catalogue, orders, payment, user) — incident window 2024-01-19 06:01:32 to 2024-01-19 06:31:32 UTC:

Per-service top KPI (highest severity)
- carts — kpi_name: latency-50; total_points_in_window=25; count_exceedances=2; earliest_exceedance=2024-01-19T06:11:00+00:00; max_value_in_window=2.161120; series_global_P95=1.263866; severity=0.709928
- catalogue — kpi_name: socket; total_points_in_window=25; count_exceedances=2; earliest_exceedance=2024-01-19T06:04:00+00:00; max_value_in_window=10.0; series_global_P95=9.493333; severity=0.053371
- orders — kpi_name: latency-90; total_points_in_window=25; count_exceedances=2; earliest_exceedance=2024-01-19T06:23:00+00:00; max_value_in_window=3.278652; series_global_P95=2.542881; severity=0.289345
- payment — kpi_name: workload; total_points_in_window=25; count_exceedances=2; earliest_exceedance=2024-01-19T06:04:00+00:00; max_value_in_window=2.194121; series_global_P95=2.071944; severity=0.058967
- user — kpi_name: workload; total_points_in_window=25; count_exceedances=2; earliest_exceedance=2024-01-19T06:16:00+00:00; max_value_in_window=18.288576; series_global_P95=17.719310; severity=0.032127

Summary counts
- total_candidate_series_examined: 36
- total_candidate_series_excluded_due_to_trivial_zero (series with P95==0 and max==0): 6

Interpretation: each candidate service has at least one non-trivial KPI with some exceedances in the window; carts (latency-50) shows the largest relative severity among these top KPIs.

The original code execution output of IPython Kernel is also provided below for reference:

(     cmdb_id    kpi_name  total_points_in_window  count_exceedances earliest_exceedance_timestamp_utc  max_value_in_window  series_global_p95  severity
0      carts  latency-50                      25                  2         2024-01-19 06:11:00+00:00             2.161120           1.263866  0.709928
1  catalogue      socket                      25                  2         2024-01-19 06:04:00+00:00            10.000000           9.493333  0.053371
2     orders  latency-90                      25                  2         2024-01-19 06:23:00+00:00             3.278652           2.542881  0.289345
3    payment    workload                      25                  2         2024-01-19 06:04:00+00:00             2.194121           2.071944  0.058967
4       user    workload                      25                  2         2024-01-19 06:16:00+00:00            18.288576          17.719310  0.032127,                                               metric  value
0                    total_candidate_series_examined     36
1  total_candidate_series_excluded_due_to_trivial...      6)```
```

In [ ]:
# In[6]:

# Focused analysis for cmdb_id == 'carts'
import pandas as pd

# Reuse metric_df (assumed loaded) and its 'dt' column (UTC)
svc = 'carts'
start_ts = pd.Timestamp("2024-01-19 06:01:32", tz="UTC")
end_ts   = pd.Timestamp("2024-01-19 06:31:32", tz="UTC")

# 1) Compute series global P95 for every (cmdb_id='carts', kpi_name) using full metric_df
p95_carts = (
    metric_df[metric_df['cmdb_id'] == svc]
    .groupby('kpi_name', as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'series_global_p95'})
)

# total kpi series examined for carts
total_kpi_series_examined_for_carts = int(p95_carts.shape[0])

# 2) Filter metric_df to the incident window for carts and merge P95
metric_window_carts = metric_df[
    (metric_df['cmdb_id'] == svc) &
    (metric_df['dt'] >= start_ts) &
    (metric_df['dt'] <= end_ts)
].copy()

# Merge P95 into window rows (on kpi_name)
metric_window_carts = metric_window_carts.merge(p95_carts, on='kpi_name', how='right')

# After merge, rows that had no points in window will have NaN in 'value'/'dt' — keep them for KPI summary
# 3) Identify exceedances (strict >)
metric_window_carts['is_exceed'] = metric_window_carts['value'] > metric_window_carts['series_global_p95']

# KPI-level aggregation (include all kpis from p95_carts)
def agg_kpi(g):
    vals = g['value'].dropna()
    total_points = int(vals.shape[0])
    count_exc = int((vals > g['series_global_p95'].iloc[0]).sum()) if total_points>0 else 0
    earliest_exc = g.loc[(g['value'] > g['series_global_p95'].iloc[0]) & (~g['dt'].isna()), 'dt'].min() if count_exc>0 else pd.NaT
    maxv = float(vals.max()) if total_points>0 else 0.0
    p95 = float(g['series_global_p95'].iloc[0])
    overall_severity = (maxv - p95) / max(1e-9, p95)
    return pd.Series({
        'total_points_in_window': total_points,
        'count_exceedances': count_exc,
        'earliest_exceedance_timestamp_utc': earliest_exc,
        'max_value_in_window': maxv,
        'series_global_p95': p95,
        'overall_severity': overall_severity
    })

kpi_summary = metric_window_carts.groupby('kpi_name', as_index=False).apply(agg_kpi).reset_index(drop=True)

# Ensure ordering (alphabetical) for compact display
kpi_summary = kpi_summary[['kpi_name','total_points_in_window','count_exceedances',
                           'earliest_exceedance_timestamp_utc','max_value_in_window','series_global_p95','overall_severity']].sort_values('kpi_name').reset_index(drop=True)

# 2/3) For exceedance rows only, group timestamps into consecutive-minute runs per kpi
exceed_df = metric_window_carts[metric_window_carts['is_exceed'] & metric_window_carts['dt'].notna()].copy()

runs = []
if not exceed_df.empty:
    # For each kpi_name, sort by dt and label runs where consecutive minutes (dt diff == 1 minute)
    def label_and_aggregate(df):
        df = df.sort_values('dt').copy()
        # compute boolean where new run starts: True when dt.diff() != 1 minute or first row
        diff = df['dt'].diff()
        new_run = (diff != pd.Timedelta(minutes=1))  # first row diff is NaT -> True
        run_id = new_run.cumsum()
        df['run_id'] = run_id
        # aggregate per run
        agg = df.groupby('run_id').agg(
            kpi_name = ('kpi_name','first'),
            run_start = ('dt','min'),
            run_end = ('dt','max'),
            run_length_in_points = ('dt','size'),
            run_max_value = ('value','max'),
            series_global_p95 = ('series_global_p95','first')
        ).reset_index(drop=True)
        return agg

    runs_df = exceed_df.groupby('kpi_name', group_keys=False).apply(label_and_aggregate).reset_index(drop=True)
    # 4) Exclude runs with run_length_in_points == 1 (isolated spikes)
    runs_df = runs_df[runs_df['run_length_in_points'] > 1].copy()
    if not runs_df.empty:
        # 3 cont) compute run_severity
        runs_df['run_severity'] = (runs_df['run_max_value'] - runs_df['series_global_p95']) / runs_df['series_global_p95'].clip(lower=1e-9)
        # reorder columns for clarity
        runs_df = runs_df[['kpi_name','run_start','run_end','run_length_in_points','run_max_value','series_global_p95','run_severity']]
        # sort by run_severity desc and limit to top 10
        runs_df = runs_df.sort_values('run_severity', ascending=False).reset_index(drop=True)
        runs_top10 = runs_df.head(10).copy()
    else:
        runs_top10 = pd.DataFrame(columns=['kpi_name','run_start','run_end','run_length_in_points','run_max_value','series_global_p95','run_severity'])
else:
    runs_top10 = pd.DataFrame(columns=['kpi_name','run_start','run_end','run_length_in_points','run_max_value','series_global_p95','run_severity'])

# Counts
total_runs_found = int(runs_top10.shape[0])  # number of non-trivial runs returned (after excluding length==1 and limited to top10)
# But also compute total non-trivial runs before top10 limit:
if 'runs_df' in locals() and not runs_df.empty:
    total_nontrivial_runs = int(runs_df.shape[0])
else:
    total_nontrivial_runs = 0

total_runs_with_severity_gt_0_5 = int(((runs_df['run_severity'] > 0.5).sum()) if ('runs_df' in locals() and not runs_df.empty) else 0)

# Prepare compact counts DataFrame
counts_df = pd.DataFrame([
    {'metric': 'total_kpi_series_examined_for_carts', 'value': total_kpi_series_examined_for_carts},
    {'metric': 'total_nontrivial_runs_found_across_kpis', 'value': total_nontrivial_runs},
    {'metric': 'total_runs_with_severity_gt_0.5', 'value': total_runs_with_severity_gt_0_5}
])

# Final compact outputs:
# A) kpi_summary (one row per carts kpi)
# B) runs_top10 (non-trivial runs sorted by run_severity desc limited to 10)
# counts_df: counts requested

# Ensure timestamp columns are timezone-aware datetimes (they are), and keep output compact
kpi_summary, runs_top10, counts_df

```
Out[6]:
```
summary = (
    "Carts metric analysis (window 2024-01-19 06:01:32 to 2024-01-19 06:31:32 UTC):\n\n"
    "A) KPI-level summary (one row per carts KPI):\n"
    "- cpu: points=25, exceedances=2, earliest_exceedance=2024-01-19T06:15:00+00:00, max=1.581104, P95=1.513572, severity=0.0446\n"
    "- error: points=25, exceedances=0, earliest_exceedance=None, max=0.0, P95=0.0, severity=0.0\n"
    "- latency-50: points=25, exceedances=2, earliest_exceedance=2024-01-19T06:11:00+00:00, max=2.161120, P95=1.263866, severity=0.7099\n"
    "- latency-90: points=25, exceedances=2, earliest_exceedance=2024-01-19T06:14:00+00:00, max=7.719347, P95=5.364331, severity=0.4390\n"
    "- mem: points=25, exceedances=2, earliest_exceedance=2024-01-19T06:25:00+00:00, max=222451500.0, P95=222439900.0, severity=0.000052\n"
    "- socket: points=25, exceedances=2, earliest_exceedance=2024-01-19T06:23:00+00:00, max=56.88333, P95=50.75333, severity=0.12078\n"
    "- workload: points=25, exceedances=2, earliest_exceedance=2024-01-19T06:15:00+00:00, max=7.809967, P95=7.774197, severity=0.00460\n\n"
    "B) Non-trivial consecutive exceedance runs (length>1), top by run severity (UTC):\n"
    "1) kpi=socket, run_start=2024-01-19T06:23:00+00:00, run_end=2024-01-19T06:24:00+00:00, "
    "run_length=2, run_max=56.88333, series_P95=50.75333, run_severity=0.12078\n\n"
    "Counts:\n"
    "- total_kpi_series_examined_for_carts: 7\n"
    "- total_nontrivial_runs_found_across_kpis: 1\n"
    "- total_runs_with_severity_gt_0.5: 0\n\n"
    "Key takeaway: latency-50 shows the highest relative severity (0.71) at the KPI level, "
    "but the only non-trivial multi-minute exceedance run observed is on socket (2 consecutive minutes) with moderate severity (0.12)."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(     kpi_name  total_points_in_window  count_exceedances earliest_exceedance_timestamp_utc  max_value_in_window  series_global_p95  overall_severity
0         cpu                      25                  2         2024-01-19 06:15:00+00:00         1.581104e+00       1.513572e+00          0.044617
1       error                      25                  0                               NaT         0.000000e+00       0.000000e+00          0.000000
2  latency-50                      25                  2         2024-01-19 06:11:00+00:00         2.161120e+00       1.263866e+00          0.709928
3  latency-90                      25                  2         2024-01-19 06:14:00+00:00         7.719347e+00       5.364331e+00          0.439014
4         mem                      25                  2         2024-01-19 06:25:00+00:00         2.224515e+08       2.224399e+08          0.000052
5      socket                      25                  2         2024-01-19 06:23:00+00:00         5.688333e+01       5.075333e+01          0.120780
6    workload                      25                  2         2024-01-19 06:15:00+00:00         7.809967e+00       7.774197e+00          0.004601,   kpi_name                 run_start                   run_end  run_length_in_points  run_max_value  series_global_p95  run_severity
0   socket 2024-01-19 06:23:00+00:00 2024-01-19 06:24:00+00:00                     2      56.883333          50.753333       0.12078,                                     metric  value
0      total_kpi_series_examined_for_carts      7
1  total_nontrivial_runs_found_across_kpis      1
2          total_runs_with_severity_gt_0.5      0)```
```